
# Notebook 17 — Residual Topology Geometry

Notebook 16 showed:

```text
shared bounded transition manifold exists after renormalized collapse
```

Notebook 17 asks:

> after collapse, is residual fragmentation itself structured?

Core claim tested here:

```text
Collapse removes the dominant transition profile but does not erase topology.
Residual fields remain structured and topology-specific.
```

Residual definition:

```text
Δ(z) = CGCS(z) − shared_profile(z)
```

where:

```text
shared_profile(z) = 1 − 1/(1 + exp(−z))
```


## Imports and setup

In [ ]:

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

GRAPH_SIZES = [16, 32, 64, 128]
TOPOLOGIES = [
    "ring_lattice",
    "small_world",
    "erdos_renyi",
    "scale_free",
    "modular_clustered",
]

print("Ready.")



## Load Notebook 16 outputs

This notebook expects Notebook 16 output files in `results/`.

Required:

```text
renormalized_collapse_data.csv
universality_window_summary.csv
critical_midpoint_scaling.csv
critical_scaling_graph_diagnostics.csv
```

If running in a fresh environment, upload or copy those files into `results/`.


In [ ]:

required_files = {
    "collapse": RESULTS_DIR / "renormalized_collapse_data.csv",
    "windows": RESULTS_DIR / "universality_window_summary.csv",
    "midpoints": RESULTS_DIR / "critical_midpoint_scaling.csv",
    "diagnostics": RESULTS_DIR / "critical_scaling_graph_diagnostics.csv",
}

missing = [str(path) for path in required_files.values() if not path.exists()]

if missing:
    print("Missing expected Notebook 16 outputs:")
    for path in missing:
        print(" -", path)
    print("\nIf files were downloaded from Colab, upload them into results/ before continuing.")
else:
    print("All expected inputs found.")


In [ ]:

collapse_df = pd.read_csv(required_files["collapse"])
window_df = pd.read_csv(required_files["windows"])
midpoint_df = pd.read_csv(required_files["midpoints"])
diag_df = pd.read_csv(required_files["diagnostics"])

# Normalize column names defensively.
collapse_df = collapse_df.replace([np.inf, -np.inf], np.nan).dropna()
window_df = window_df.replace([np.inf, -np.inf], np.nan).dropna()
midpoint_df = midpoint_df.replace([np.inf, -np.inf], np.nan).dropna()
diag_df = diag_df.replace([np.inf, -np.inf], np.nan).dropna()

collapse_df.head()


## Shared profile and residual field

In [ ]:

def logistic_z(z):
    return 1 / (1 + np.exp(-z))

def shared_profile(z):
    return 1 - logistic_z(z)

# Recompute residuals defensively in case input columns differ.
collapse_df["shared_profile_recomputed"] = shared_profile(collapse_df["z"].to_numpy())
collapse_df["residual"] = collapse_df["cgcs"] - collapse_df["shared_profile_recomputed"]
collapse_df["abs_residual"] = collapse_df["residual"].abs()
collapse_df["residual_energy"] = collapse_df["residual"] ** 2

collapse_df.to_csv(
    RESULTS_DIR / "residual_field_data.csv",
    index=False
)

collapse_df.head()


## Residual field overview

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
axes = axes.ravel()

for ax, topology in zip(axes, TOPOLOGIES):
    sub_topo = collapse_df[collapse_df["topology"] == topology]

    for N in GRAPH_SIZES:
        sub = sub_topo[sub_topo["n_modules"] == N]
        if len(sub) == 0:
            continue

        ax.scatter(
            sub["z"],
            sub["residual"],
            s=20,
            alpha=0.45,
            label=f"N={N}"
        )

    ax.axhline(0, color="black", linewidth=1, linestyle="--")
    ax.set_title(topology.replace("_", " "))
    ax.set_xlabel("renormalized z")
    ax.set_ylabel("residual Δ(z)")
    ax.set_xlim(-6, 6)
    ax.grid(alpha=0.3)

axes[-1].axis("off")
axes[0].legend(fontsize=8)

plt.tight_layout()

fig_path = FIG_DIR / "residual_field_by_topology.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Residual localization

In [ ]:

localization_rows = []

for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
    energy = sub["residual_energy"].to_numpy()
    abs_res = sub["abs_residual"].to_numpy()

    total_energy = float(np.sum(energy))
    mean_abs = float(np.mean(abs_res))
    max_abs = float(np.max(abs_res))

    if len(energy) == 0 or total_energy <= 0:
        top_10pct_concentration = 0.0
    else:
        k = max(1, int(np.ceil(0.10 * len(energy))))
        top_energy = np.sort(energy)[-k:].sum()
        top_10pct_concentration = float(top_energy / total_energy)

    localization_rows.append({
        "topology": topology,
        "n_modules": int(N),
        "mean_abs_residual": mean_abs,
        "max_abs_residual": max_abs,
        "total_residual_energy": total_energy,
        "energy_concentration_top_10pct": top_10pct_concentration,
    })

localization_df = pd.DataFrame(localization_rows)

loc_path = RESULTS_DIR / "residual_localization_summary.csv"
localization_df.to_csv(loc_path, index=False)

print(f"saved: {loc_path}")
localization_df.head()


In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = localization_df[localization_df["topology"] == topology]

    plt.plot(
        sub["n_modules"],
        sub["energy_concentration_top_10pct"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.xlabel("graph size N")
plt.ylabel("top 10% residual-energy concentration")
plt.title("Residual energy localization")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "residual_energy_localization.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Residual symmetry and asymmetry

In [ ]:

symmetry_rows = []

for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
    left = sub[sub["z"] < 0]
    right = sub[sub["z"] >= 0]

    left_energy = float(left["residual_energy"].sum())
    right_energy = float(right["residual_energy"].sum())
    total_energy = left_energy + right_energy

    if total_energy > 0:
        asymmetry = float((right_energy - left_energy) / total_energy)
    else:
        asymmetry = 0.0

    symmetry_rows.append({
        "topology": topology,
        "n_modules": int(N),
        "left_energy": left_energy,
        "right_energy": right_energy,
        "total_energy": total_energy,
        "asymmetry": asymmetry,
    })

symmetry_df = pd.DataFrame(symmetry_rows)

sym_path = RESULTS_DIR / "residual_symmetry_summary.csv"
symmetry_df.to_csv(sym_path, index=False)

print(f"saved: {sym_path}")
symmetry_df.head()


In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = symmetry_df[symmetry_df["topology"] == topology]

    plt.plot(
        sub["n_modules"],
        sub["asymmetry"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("graph size N")
plt.ylabel("residual asymmetry")
plt.title("Residual asymmetry by topology")
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "residual_asymmetry_by_topology.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")



## Residual curvature

Curvature identifies where topology bends away from the shared manifold.

We estimate:

```text
d²Δ / dz²
```

on interpolated uniform z grids.


In [ ]:

def interpolate_residual(sub, z_grid):
    ordered = sub.sort_values("z")
    z = ordered["z"].to_numpy(dtype=float)
    r = ordered["residual"].to_numpy(dtype=float)

    # Deduplicate z if needed.
    tmp = pd.DataFrame({"z": z, "r": r}).groupby("z", as_index=False).mean()
    z = tmp["z"].to_numpy()
    r = tmp["r"].to_numpy()

    if len(z) < 4:
        return np.full_like(z_grid, np.nan)

    return np.interp(z_grid, z, r, left=np.nan, right=np.nan)

z_grid = np.linspace(-6, 6, 241)
curvature_rows = []
curvature_profile_rows = []

for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
    r_grid = interpolate_residual(sub, z_grid)
    mask = np.isfinite(r_grid)

    if mask.sum() < 5:
        continue

    z_valid = z_grid[mask]
    r_valid = r_grid[mask]

    first = np.gradient(r_valid, z_valid)
    second = np.gradient(first, z_valid)

    mean_abs_curvature = float(np.mean(np.abs(second)))
    max_abs_curvature = float(np.max(np.abs(second)))
    curvature_energy = float(np.sum(second ** 2))

    curvature_rows.append({
        "topology": topology,
        "n_modules": int(N),
        "mean_abs_curvature": mean_abs_curvature,
        "max_abs_curvature": max_abs_curvature,
        "curvature_energy": curvature_energy,
    })

    for z_val, curv_val in zip(z_valid, second):
        curvature_profile_rows.append({
            "topology": topology,
            "n_modules": int(N),
            "z": float(z_val),
            "residual_curvature": float(curv_val),
            "abs_residual_curvature": float(abs(curv_val)),
        })

curvature_df = pd.DataFrame(curvature_rows)
curvature_profile_df = pd.DataFrame(curvature_profile_rows)

curv_path = RESULTS_DIR / "residual_curvature_summary.csv"
curv_profile_path = RESULTS_DIR / "residual_curvature_profiles.csv"

curvature_df.to_csv(curv_path, index=False)
curvature_profile_df.to_csv(curv_profile_path, index=False)

print(f"saved: {curv_path}")
print(f"saved: {curv_profile_path}")
curvature_df.head()


In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
axes = axes.ravel()

for ax, topology in zip(axes, TOPOLOGIES):
    sub_topo = curvature_profile_df[curvature_profile_df["topology"] == topology]

    for N in GRAPH_SIZES:
        sub = sub_topo[sub_topo["n_modules"] == N]
        if len(sub) == 0:
            continue

        ax.plot(
            sub["z"],
            sub["residual_curvature"],
            linewidth=1.5,
            label=f"N={N}"
        )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(topology.replace("_", " "))
    ax.set_xlabel("renormalized z")
    ax.set_ylabel("d²Δ/dz²")
    ax.set_xlim(-6, 6)
    ax.grid(alpha=0.3)

axes[-1].axis("off")
axes[0].legend(fontsize=8)

plt.tight_layout()

fig_path = FIG_DIR / "residual_curvature_profiles.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Residual entropy

In [ ]:

def normalized_entropy_from_energy(z, energy, bins=24):
    z = np.asarray(z)
    energy = np.asarray(energy)

    hist, _ = np.histogram(z, bins=bins, range=(-6, 6), weights=energy)
    total = hist.sum()

    if total <= 0:
        return 0.0

    p = hist / total
    p = p[p > 0]

    return float(-np.sum(p * np.log(p)) / np.log(bins))

entropy_rows = []

for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
    entropy = normalized_entropy_from_energy(
        sub["z"].to_numpy(),
        sub["residual_energy"].to_numpy(),
        bins=24
    )

    entropy_rows.append({
        "topology": topology,
        "n_modules": int(N),
        "residual_entropy": entropy,
    })

entropy_df = pd.DataFrame(entropy_rows)

entropy_path = RESULTS_DIR / "residual_entropy_summary.csv"
entropy_df.to_csv(entropy_path, index=False)

print(f"saved: {entropy_path}")
entropy_df.head()


In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = entropy_df[entropy_df["topology"] == topology]

    plt.plot(
        sub["n_modules"],
        sub["residual_entropy"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.xlabel("graph size N")
plt.ylabel("normalized residual entropy")
plt.ylim(0, 1.05)
plt.title("Residual entropy by topology")
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "residual_entropy_by_topology.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Residual spectral structure

In [ ]:

spectral_rows = []
spectral_profile_rows = []

z_fft = np.linspace(-6, 6, 256)

for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
    r_grid = interpolate_residual(sub, z_fft)

    if not np.isfinite(r_grid).all():
        # Replace missing edges with zeros after mean-centering valid region.
        valid = np.isfinite(r_grid)
        if valid.sum() < 8:
            continue
        fill = np.nanmean(r_grid)
        r_grid = np.where(valid, r_grid, fill)

    r_centered = r_grid - np.mean(r_grid)
    fft_vals = np.fft.rfft(r_centered)
    mag = np.abs(fft_vals)
    power = mag ** 2

    freqs = np.fft.rfftfreq(len(r_centered), d=(z_fft[1] - z_fft[0]))

    # Exclude DC mode for structure metrics.
    non_dc_power = power.copy()
    non_dc_power[0] = 0

    total_power = float(non_dc_power.sum())

    if total_power <= 0:
        low_energy = 0.0
        high_energy = 0.0
        spectral_ratio = 0.0
        dominant_mode = 0
    else:
        cutoff = max(2, int(0.20 * len(non_dc_power)))
        low_energy = float(non_dc_power[1:cutoff].sum() / total_power)
        high_energy = float(non_dc_power[cutoff:].sum() / total_power)
        spectral_ratio = float(high_energy / max(low_energy, 1e-9))
        dominant_mode = int(np.argmax(non_dc_power))

    spectral_rows.append({
        "topology": topology,
        "n_modules": int(N),
        "low_frequency_energy": low_energy,
        "high_frequency_energy": high_energy,
        "spectral_ratio": spectral_ratio,
        "dominant_mode": dominant_mode,
        "total_spectral_power": total_power,
    })

    for f, m in zip(freqs, mag):
        spectral_profile_rows.append({
            "topology": topology,
            "n_modules": int(N),
            "frequency": float(f),
            "magnitude": float(m),
        })

spectral_df = pd.DataFrame(spectral_rows)
spectral_profile_df = pd.DataFrame(spectral_profile_rows)

spectral_path = RESULTS_DIR / "residual_spectral_summary.csv"
spectral_profile_path = RESULTS_DIR / "residual_spectral_profiles.csv"

spectral_df.to_csv(spectral_path, index=False)
spectral_profile_df.to_csv(spectral_profile_path, index=False)

print(f"saved: {spectral_path}")
print(f"saved: {spectral_profile_path}")
spectral_df.head()


In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
axes = axes.ravel()

for ax, topology in zip(axes, TOPOLOGIES):
    sub_topo = spectral_profile_df[spectral_profile_df["topology"] == topology]

    for N in GRAPH_SIZES:
        sub = sub_topo[
            (sub_topo["n_modules"] == N)
            & (sub_topo["frequency"] > 0)
            & (sub_topo["frequency"] < 4)
        ]

        if len(sub) == 0:
            continue

        ax.plot(
            sub["frequency"],
            sub["magnitude"],
            linewidth=1.5,
            label=f"N={N}"
        )

    ax.set_title(topology.replace("_", " "))
    ax.set_xlabel("frequency")
    ax.set_ylabel("FFT magnitude")
    ax.grid(alpha=0.3)

axes[-1].axis("off")
axes[0].legend(fontsize=8)

plt.tight_layout()

fig_path = FIG_DIR / "residual_spectral_profiles.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = spectral_df[spectral_df["topology"] == topology]

    plt.plot(
        sub["n_modules"],
        sub["spectral_ratio"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.xlabel("graph size N")
plt.ylabel("high / low frequency residual energy")
plt.title("Residual spectral ratio by topology")
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "residual_spectral_ratio_by_topology.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Residual persistence vs graph size

In [ ]:

persistence_df = (
    localization_df
    .merge(symmetry_df[["topology", "n_modules", "asymmetry"]], on=["topology", "n_modules"])
    .merge(entropy_df, on=["topology", "n_modules"])
    .merge(spectral_df[["topology", "n_modules", "spectral_ratio"]], on=["topology", "n_modules"])
    .merge(curvature_df[["topology", "n_modules", "curvature_energy"]], on=["topology", "n_modules"])
)

persistence_path = RESULTS_DIR / "residual_persistence_vs_size.csv"
persistence_df.to_csv(persistence_path, index=False)

print(f"saved: {persistence_path}")
persistence_df.head()


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.ravel()

metrics = [
    ("mean_abs_residual", "mean |residual|"),
    ("residual_entropy", "residual entropy"),
    ("asymmetry", "asymmetry"),
    ("spectral_ratio", "spectral ratio"),
]

for ax, (metric, ylabel) in zip(axes, metrics):
    for topology in TOPOLOGIES:
        sub = persistence_df[persistence_df["topology"] == topology]

        ax.plot(
            sub["n_modules"],
            sub[metric],
            marker="o",
            linewidth=2,
            label=topology.replace("_", " ")
        )

    ax.set_title(ylabel)
    ax.set_xlabel("graph size N")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)

axes[0].legend(fontsize=8)

plt.tight_layout()

fig_path = FIG_DIR / "residual_persistence_vs_graph_size.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Residual geometry map

In [ ]:

geometry_df = persistence_df.copy()

geometry_path = RESULTS_DIR / "residual_geometry_features.csv"
geometry_df.to_csv(geometry_path, index=False)

print(f"saved: {geometry_path}")
geometry_df.head()


In [ ]:

plt.figure(figsize=(10, 7))

for topology in TOPOLOGIES:
    sub = geometry_df[geometry_df["topology"] == topology]

    sizes = 80 + 500 * (
        sub["curvature_energy"] / max(geometry_df["curvature_energy"].max(), 1e-9)
    )

    plt.scatter(
        sub["residual_entropy"],
        sub["asymmetry"],
        s=sizes,
        alpha=0.75,
        label=topology.replace("_", " ")
    )

    for _, row in sub.iterrows():
        plt.annotate(
            f"N={int(row['n_modules'])}",
            xy=(row["residual_entropy"], row["asymmetry"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
        )

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("residual entropy")
plt.ylabel("residual asymmetry")
plt.title("Residual geometry map\n(size = curvature energy)")
plt.grid(alpha=0.3)
plt.legend(fontsize=8)

fig_path = FIG_DIR / "residual_geometry_map.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Summary export

In [ ]:

summary_by_topology = []

for topology in TOPOLOGIES:
    sub = geometry_df[geometry_df["topology"] == topology]

    if len(sub) == 0:
        continue

    summary_by_topology.append({
        "topology": topology,
        "mean_abs_residual": float(sub["mean_abs_residual"].mean()),
        "mean_entropy": float(sub["residual_entropy"].mean()),
        "mean_asymmetry": float(sub["asymmetry"].mean()),
        "mean_spectral_ratio": float(sub["spectral_ratio"].mean()),
        "mean_curvature_energy": float(sub["curvature_energy"].mean()),
    })

summary_topology_df = pd.DataFrame(summary_by_topology)

summary_topology_path = RESULTS_DIR / "residual_topology_summary_by_topology.csv"
summary_topology_df.to_csv(summary_topology_path, index=False)

if len(summary_topology_df) > 0:
    most_localized = localization_df.sort_values(
        "energy_concentration_top_10pct",
        ascending=False
    ).iloc[0]

    highest_entropy = entropy_df.sort_values(
        "residual_entropy",
        ascending=False
    ).iloc[0]

    highest_spectral_ratio = spectral_df.sort_values(
        "spectral_ratio",
        ascending=False
    ).iloc[0]
else:
    most_localized = None
    highest_entropy = None
    highest_spectral_ratio = None

summary = {
    "notebook": "17_residual_topology_geometry.ipynb",

    "core_claim": (
        "Residuals around the shared bounded transition manifold are structured, "
        "topology-specific, and measurable through localization, asymmetry, "
        "curvature, entropy, and spectral diagnostics."
    ),

    "interpretation": (
        "The shared transition persists; topology reappears as structured "
        "residual geometry."
    ),

    "most_localized_residual": None if most_localized is None else {
        "topology": most_localized["topology"],
        "n_modules": int(most_localized["n_modules"]),
        "energy_concentration_top_10pct": float(most_localized["energy_concentration_top_10pct"]),
    },

    "highest_residual_entropy": None if highest_entropy is None else {
        "topology": highest_entropy["topology"],
        "n_modules": int(highest_entropy["n_modules"]),
        "residual_entropy": float(highest_entropy["residual_entropy"]),
    },

    "highest_spectral_ratio": None if highest_spectral_ratio is None else {
        "topology": highest_spectral_ratio["topology"],
        "n_modules": int(highest_spectral_ratio["n_modules"]),
        "spectral_ratio": float(highest_spectral_ratio["spectral_ratio"]),
    },

    "figures": [
        "residual_field_by_topology.png",
        "residual_energy_localization.png",
        "residual_asymmetry_by_topology.png",
        "residual_curvature_profiles.png",
        "residual_entropy_by_topology.png",
        "residual_spectral_profiles.png",
        "residual_spectral_ratio_by_topology.png",
        "residual_persistence_vs_graph_size.png",
        "residual_geometry_map.png",
    ],

    "results": [
        "residual_field_data.csv",
        "residual_localization_summary.csv",
        "residual_symmetry_summary.csv",
        "residual_curvature_summary.csv",
        "residual_curvature_profiles.csv",
        "residual_entropy_summary.csv",
        "residual_spectral_summary.csv",
        "residual_spectral_profiles.csv",
        "residual_persistence_vs_size.csv",
        "residual_geometry_features.csv",
        "residual_topology_summary_by_topology.csv",
        "residual_topology_geometry_summary.json",
    ],
}

summary_path = RESULTS_DIR / "residual_topology_geometry_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

doc_lines = [
    "# Notebook 17 — Residual Topology Geometry",
    "",
    "**Core claim:** residuals around the shared bounded transition manifold are structured, topology-specific, and measurable.",
    "",
    "Short interpretation:",
    "",
    "> The shared transition persists; topology reappears as structured residual geometry.",
    "",
    "Main outputs:",
    "",
    "- `figures/residual_field_by_topology.png`",
    "- `figures/residual_energy_localization.png`",
    "- `figures/residual_asymmetry_by_topology.png`",
    "- `figures/residual_curvature_profiles.png`",
    "- `figures/residual_entropy_by_topology.png`",
    "- `figures/residual_spectral_profiles.png`",
    "- `figures/residual_spectral_ratio_by_topology.png`",
    "- `figures/residual_persistence_vs_graph_size.png`",
    "- `figures/residual_geometry_map.png`",
    "",
]

doc_path = DOCS_DIR / "notebook_17_residual_topology_geometry.md"
doc_path.write_text("\n".join(doc_lines), encoding="utf-8")

print(json.dumps(summary, indent=2))
print(f"saved: {summary_path}")
print(f"saved: {doc_path}")
print(f"saved: {summary_topology_path}")



## Final interpretation

Notebook 17 tests whether collapse residuals carry structure.

Careful conclusion:

```text
Collapse removes the dominant transition profile but does not erase topology.
Residual fields remain structured and topology-specific, showing organized
fragmentation around the shared bounded manifold.
```

Short version:

```text
The shared transition persists; topology reappears as structured residual geometry.
```


## Optional export zip

In [ ]:

zip_path = Path("notebook_17_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file():
                    zf.write(file)

print(f"Created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))
